## Why AI at Nexora?
I'm Kartike Verma, a final year student at Amity University, Jaipur, Rajasthan, and currently pursuing B.Tech CSE. I have a keen interest in Data Science, AI/ML, and have good analytical and technical skills with experience in IoT, Machine Learning. I want to be an AI intern at Nexora with my good skills in Python and frameworks like scikit-learn, TensorFlow, and pandas, and I want to be a part of Nexora to solve some real-world problems related to AI.

In [93]:
import pandas as pd,numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

In [63]:
items = [
     {"id": 1, "name": "Boho Dress", "desc": "Flowy maxi dress, soft fabrics and earthy hues for carefree bohemian festivals", "tags": ["boho","festival","flowy"]},
  {"id": 2, "name": "Urban Bomber", "desc": "Cropped bomber jacket, reflective finish for bold and energetic city nights", "tags": ["urban","edgy","night"]},
  {"id": 3, "name": "Cozy Knit Sweater", "desc": "Oversized knit sweater, warm and plush for relaxed coffee dates and cozy evenings", "tags": ["cozy","casual","warm"]},
  {"id": 4, "name": "Office Blazer", "desc": "Tailored leather blazer, sleek and sharp for confident and modern business looks", "tags": ["formal","sharp","modern"]},
  {"id": 5, "name": "Sporty Runner", "desc": "Lightweight running jacket, breathable and dynamic for active urban mornings", "tags": ["sporty","active","breathable"]}
]
df = pd.DataFrame(items)
df

,id,name,desc,tags
0,1,Boho Dress,"Flowy maxi dress, soft fabrics and earthy hues...","[boho, festival, flowy]"
1,2,Urban Bomber,"Cropped bomber jacket, reflective finish for b...","[urban, edgy, night]"
2,3,Cozy Knit Sweater,"Oversized knit sweater, warm and plush for rel...","[cozy, casual, warm]"
3,4,Office Blazer,"Tailored leather blazer, sleek and sharp for c...","[formal, sharp, modern]"
4,5,Sporty Runner,"Lightweight running jacket, breathable and dyn...","[sporty, active, breathable]"


In [104]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [112]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def embed_texts(texts, batch_size=32, max_length=256):
    if isinstance(texts, str):
        texts = [texts]  
    
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
        last_hidden = outputs.last_hidden_state                 
        mask = encoded["attention_mask"].unsqueeze(-1).type_as(last_hidden)  

        summed = (last_hidden * mask).sum(dim=1)                
        counts = mask.sum(dim=1).clamp(min=1e-9)                
        mean_pooled = summed / counts                          

        normalized = F.normalize(mean_pooled, p=2, dim=1)      
        all_embeds.append(normalized.cpu().numpy())

    if len(all_embeds) == 0:
        return np.zeros((0, model.config.hidden_size))         
    return np.vstack(all_embeds)                               

df_texts = df['desc'].astype(str).tolist()
all_df_embeddings = embed_texts(df_texts, batch_size=16)     
df['embeddings'] = list(all_df_embeddings)


In [106]:
def vibe_checker(user_emeddings, desc_embedding):
    similarity_scores = cosine_similarity(array_embeddings_user, all_df_embeddings)
    print("\nSimilarity Scores:")
    for i, score in enumerate(similarity_scores[0]):
        print(f"Row {i} ('{df['desc'].iloc[i]}'): {score:.4f}")
        

In [129]:
user_input_texts = [
    "energetic urban chic",
    "cozy countryside autumn",
    "minimalist pastel aesthetic",
]

user_embeddings = embed_texts(user_input_texts, batch_size=8)
sim_matrix = cosine_similarity(user_embeddings, all_df_embeddings)
for qi, q in enumerate(user_input_texts):
    scores = sim_matrix[qi]
    best_idx = int(np.argmax(scores))
    score = "good" if scores[best_idx] > 0.7 else "not sufficient but best in"

    print(f"\nQuery: {q}")
    print(f"{score} ('{df['desc'].iloc[best_idx]}') : {scores[best_idx]:.4f}")




Query: energetic urban chic
good ('Lightweight running jacket, breathable and dynamic for active urban mornings') : 0.7353

Query: cozy countryside autumn
good ('Oversized knit sweater, warm and plush for relaxed coffee dates and cozy evenings') : 0.7034

Query: minimalist pastel aesthetic
not sufficient but best in ('Flowy maxi dress, soft fabrics and earthy hues for carefree bohemian festivals') : 0.6744
